# Кластеризация и визуализация сообщений

Кластеризация сообщений на основе sentence embeddings, снижение размерности и интерактивная визуализация результатов.

In [1]:
import pandas as pd
import json
from typing import *
from sentence_transformers import SentenceTransformer
import umap
from sklearn.cluster import DBSCAN, HDBSCAN
import textwrap
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import re
import os

W0409 18:44:24.130000 26588 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
seed = 42
np.random.seed(seed)

In [3]:
def extract_text(msg):
    if not isinstance(msg, list):
        return msg

    result_text = ''
    for item in msg:
        if isinstance(item, str):
            result_text += (item + " ")
        else:
            result_text += (item['text'] + " ")
    return result_text

In [4]:
def get_dataframe_messages(user_id: List[str], username: List[str], date: List[str], text: List[str]):
    dataframe = pd.DataFrame({
        'user_id': user_id,
        'username': username,
        'date': date,
        'text': text,
    })
    return dataframe

In [5]:
def prepoccess_dataset(df):
    preproccessed_df = df.copy()
    preproccessed_df = df.dropna(subset=['text'])
    preproccessed_df.reset_index(drop=True, inplace=True)
    preproccessed_df['text'] = preproccessed_df['text'].map(lambda x: extract_text(x))
    preproccessed_df['text_clean_nums'] = preproccessed_df['text'].map(lambda x: re.sub(r'\d', '', x))
    preproccessed_df = preproccessed_df[preproccessed_df['text_clean_nums'] != '']
    return preproccessed_df

In [6]:
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
import os
import numpy as np

def vizualize_df(df, labels, save_path=None):
    fig = px.scatter(
        df, 
        x='x', 
        y='y', 
        hover_data=['text', 'user', 'date', 'time'], 
        color=labels,
        custom_data=[df.index]
    )

    fig.update_traces(marker=dict(size=3))
    fig.update_layout(
        width=800,
        height=600,
        dragmode="pan",
        clickmode='event+select',
        modebar_add=['drawopenpath', 'eraseshape'],
        newshape_line=dict(color='red', width=2),
        margin=dict(t=30, b=0, l=0, r=0)
    )

    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode='markers',
        marker=dict(size=8, color='red', line=dict(width=1, color='black')),
        name='__highlight__',
        showlegend=False,
        hoverinfo='skip'
    ))

    f = go.FigureWidget(fig)
    f._config = {'scrollZoom': True, 'displayModeBar': True}

    highlight_idx = len(f.data) - 1

    table_layout = widgets.Layout(
        height='600px', width='100%',
        overflow_y='auto', overflow_x='auto',
        border='1px solid lightgray'
    )

    out_table = widgets.Output(layout=table_layout)
    label_table = widgets.Output(layout=table_layout)
    out_text = widgets.Output()

    tabs = widgets.Tab(children=[out_table, label_table])
    tabs.set_title(0, 'Выделение (Lasso)')
    tabs.set_title(1, 'Поиск (Label)')

    def selection_fn(trace, points, selector):
        if not points.point_inds:
            return
        df_without_br = df.copy()
        df_without_br['text'] = df['text'].str.replace('<br>', ' ', regex=False)
        df_without_br['label'] = labels

        tabs.selected_index = 0
        selected_indices = [trace.customdata[i][0] for i in points.point_inds]
        selected_df = df_without_br.drop(columns=['x', 'y']).loc[selected_indices]
        selected_df = selected_df[['text', 'user', 'label', 'date', 'time']]

        with out_table:
            out_table.clear_output()
            print(f"Выделено {len(selected_df)} точек:")
            display(selected_df.style.set_properties(
                **{'white-space': 'pre-wrap', 'text-align': 'left'}
            ))

        with out_text:
            out_text.clear_output()
            string = ' <sep> '.join(selected_df['text'].tolist())
            print("Текст выделенных точек:")
            print(string[:1000] + "..." if len(string) > 1000 else string)

    for trace in f.data:
        if trace.name != '__highlight__':
            trace.on_selection(selection_fn)


    labels_array = np.array(labels)
    
    def select_cluster(label):
        mask = labels_array == label
        
        if not mask.any():
            with label_table:
                label_table.clear_output()
                print(f"Кластер {label} пуст или не существует.")
            # Очищаем подсветку
            with f.batch_update():
                f.data[highlight_idx].x = [None]
                f.data[highlight_idx].y = [None]
            tabs.selected_index = 1
            return

        sel_x = df['x'].values[mask]
        sel_y = df['y'].values[mask]

        with f.batch_update():
            f.data[highlight_idx].x = sel_x
            f.data[highlight_idx].y = sel_y

        df_without_br = df.copy()
        df_without_br['text'] = df['text'].str.replace('<br>', ' ', regex=False)
        selected_df = df_without_br.loc[mask, ['text', 'user', 'date', 'time']]

        tabs.selected_index = 1
        with label_table:
            label_table.clear_output()
            print(f"Кластер: {label} (Всего: {len(selected_df)})")
            display(selected_df.style.set_properties(
                **{'white-space': 'pre-wrap', 'text-align': 'left'}
            ))

    annotations = []
    last_click = {'x': None, 'y': None}

    def click_fn(trace, points, state):
        if points.xs and points.ys:
            last_click['x'] = points.xs[0]
            last_click['y'] = points.ys[0]

    for trace in f.data:
        if trace.name != '__highlight__':
            trace.on_click(click_fn)

    annotation_text = widgets.Text(
        placeholder='Текст подписи',
        layout=widgets.Layout(width='auto', flex='1 1 auto')
    )
    add_btn = widgets.Button(description='Добавить', button_style='primary', layout=widgets.Layout(width='100px'))
    delete_btn = widgets.Button(description='Удалить', button_style='danger', layout=widgets.Layout(width='100px'))

    def add_annotation(_):
        if last_click['x'] is None or not annotation_text.value: return
        ann = dict(
            x=last_click['x'], y=last_click['y'], text=annotation_text.value,
            showarrow=True, arrowhead=2, ax=20, ay=-20,
            bgcolor='rgba(255,255,255,0.9)', bordercolor='black'
        )
        annotations.append(ann)
        f.layout.annotations = annotations
        annotation_text.value = ''

    def delete_annotation(_):
        if annotations:
            annotations.pop()
            f.layout.annotations = annotations

    add_btn.on_click(add_annotation)
    delete_btn.on_click(delete_annotation)
    annotation_box = widgets.HBox([annotation_text, add_btn, delete_btn], layout=widgets.Layout(margin='10px 0 0 0'))

    label_input = widgets.IntText(description='Label:', layout=widgets.Layout(width='150px'))
    label_btn = widgets.Button(description='Показать', button_style='info')
    prev_label_btn = widgets.Button(description='<', layout=widgets.Layout(width='40px'))
    next_label_btn = widgets.Button(description='>', layout=widgets.Layout(width='40px'))
    reset_view_btn = widgets.Button(description='Сброс вида', button_style='warning', layout=widgets.Layout(width='100px'))

    def on_reset_click(_):
        """Просто убираем оверлей"""
        with f.batch_update():
            f.data[highlight_idx].x = [None]
            f.data[highlight_idx].y = [None]

    reset_view_btn.on_click(on_reset_click)

    def on_label_btn_click(_):
        select_cluster(label_input.value)

    def on_prev_click(_):
        label_input.value -= 1
        select_cluster(label_input.value)

    def on_next_click(_):
        label_input.value += 1
        select_cluster(label_input.value)

    label_btn.on_click(on_label_btn_click)
    prev_label_btn.on_click(on_prev_click)
    next_label_btn.on_click(on_next_click)

    label_box = widgets.HBox(
        [label_input, label_btn, prev_label_btn, next_label_btn, reset_view_btn],
        layout=widgets.Layout(margin='10px 0 0 0')
    )

    save_path_input = widgets.Text(
        value=r'../../data/graphics/plot.html',
        placeholder='Путь...', description='Save to:',
        layout=widgets.Layout(width='auto', flex='1 1 auto')
    )
    save_html_btn = widgets.Button(description='Сохранить HTML', button_style='success')
    save_status = widgets.Output()

    def save_html(_):
        path = save_path_input.value.strip()
        if not path:
            with save_status:
                save_status.clear_output()
                print('❌ Путь не задан')
            return
        dir_name = os.path.dirname(path)
        if dir_name and not os.path.exists(dir_name):
            with save_status:
                save_status.clear_output()
                print(f'❌ Директория не существует: {dir_name}')
            return
        try:
            layout_dict = f.layout.to_plotly_json()
            def clean_recursive(item):
                if isinstance(item, dict):
                    return {k: clean_recursive(v) for k, v in item.items()
                            if not str(v).startswith('<object object')}
                elif isinstance(item, list):
                    return [clean_recursive(i) for i in item]
                return item

            fig_to_save = go.Figure(data=f.data, layout=clean_recursive(layout_dict))
            fig_to_save.write_html(path, include_plotlyjs='cdn', config={'scrollZoom': True})
            with save_status:
                save_status.clear_output()
                print(f'✅ HTML сохранён: {path}')
        except Exception as e:
            with save_status:
                save_status.clear_output()
                print(f'❌ Ошибка: {e}')

    save_html_btn.on_click(save_html)
    save_box = widgets.VBox([
        widgets.HBox([save_path_input, save_html_btn]), save_status
    ], layout=widgets.Layout(margin='20px 0 0 0', border='1px solid #eee', padding='5px'))

    left_box = widgets.VBox(
        [f, label_box, annotation_box, save_box, out_text],
        layout=widgets.Layout(width='60%', margin='0 10px 0 0')
    )
    right_box = widgets.VBox([tabs], layout=widgets.Layout(width='40%'))

    display(widgets.HBox([left_box, right_box], layout=widgets.Layout(align_items='flex-start')))

    if save_path is not None:
        fig.write_html(save_path, include_plotlyjs="cdn", config={"scrollZoom": True})

In [7]:
def save_txt(texts, labels, users, dates, times, path):
    df = pd.DataFrame({
        'text': texts,
        'label': labels,
        'user': users,
        'date': dates,
        'time': times
    })

    text_width = 50  

    grouped = df.groupby('label')

    with open(path, 'w', encoding='utf-8') as f:
        for label, group in grouped:
            f.write('-' * 33 + '\n')
            f.write(f"Cluster: {label}\n")
            for idx, row in enumerate(group.itertuples(index=False)):
                f.write(f'{idx+1}) {row.text:<{text_width}} | {row.user} | {row.date} {row.time}\n')

In [8]:
def text_to_scatter(text_data: List[str], clean_text: List[str], dates: List[str], users: List[str], emb_model: SentenceTransformer, clust_model, reducer, save_path=None, path_txt=None):
    embeddings = emb_model.encode(clean_text, convert_to_numpy=True)
    
    umap_emb = reducer.fit_transform(embeddings)

    formatted_text_data = list(map(lambda x: '<br>'.join(textwrap.wrap(str(x), width=40)), text_data))

    formatted_dates = []
    times = []
    for i in range(len(dates)):
        date, time = dates[i].split('T')
        date = date.split('-')
        date = f'{date[2]}-{date[1]}-{date[0]}'
        formatted_dates.append(date)
        times.append(time)
    
    
    df = pd.DataFrame({
        'x': umap_emb[:, 0],
        'y': umap_emb[:, 1],
        'text': formatted_text_data,
        'date': formatted_dates,
        'time': times,
        'user': users
    })
    
    labels = clust_model.fit_predict(umap_emb)

    if path_txt is not None:
        save_txt(text_data, labels, users, formatted_dates, times, path_txt)
        
    
    vizualize_df(df, labels, save_path)

In [9]:
def chat_history_to_scatter(json_path, emb_model, clust_model, reducer, save_path=None, path_txt=None):
    with open(json_path, 'r', encoding='utf-8') as f:
        data_json = json.load(f)

    user_id = []
    usernames = []
    dates = []
    texts = []
    for msg in data_json['messages']:
        if msg['type'] == 'message':
            user_id.append(msg['from_id'])
            if 'forwarded_from' in msg.keys():
                usernames.append(msg['forwarded_from'])
            else:
                usernames.append(msg['from'])
            dates.append(msg['date'])
            texts.append(msg['text'])

    df = get_dataframe_messages(user_id, usernames, dates, texts)
    proccessed_df = prepoccess_dataset(df)

    proccessed_df = proccessed_df.drop_duplicates(subset=['text'])
    proccessed_df.reset_index(drop=True, inplace=True)
    
    text_data = proccessed_df['text']
    text_data = text_data.astype(str).tolist()
    dates = proccessed_df['date']
    users = proccessed_df['username']
    clean_text = proccessed_df['text_clean_nums'].astype(str).tolist()

    text_to_scatter(text_data, clean_text, dates, users, emb_model, clust_model, reducer, save_path, path_txt)

# Пример использования

In [10]:
json_lera = r'../../data/chats/lera.json'
emb_model = SentenceTransformer('google/embeddinggemma-300m')
dbscan = DBSCAN(eps=0.05, min_samples=3, metric='cosine')
hdbscan = HDBSCAN(min_cluster_size=3, min_samples=2)
reducer = umap.UMAP(n_components=2, n_neighbors=4, min_dist=0, metric='cosine', random_state=seed, )
save_lera = r'../../data/graphics/lera.html'
txt_path = r'../../data/clusters/test1.txt'

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Denis\.cache\huggingface\hub\models--google--embeddinggemma-300m. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

In [11]:
chat_history_to_scatter(json_lera, emb_model, hdbscan, reducer)

C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


    'data': [{'customdata': array([[0, 'Мяу', 'Val55555ra👍', '27-…